## `ip link` — Interface State & Link-Layer Control

**Support context:** `ip addr` shows you IPs, `ip route` shows you where traffic goes once it has an IP — `ip link` sits a layer below both. It's the link/device layer: is this interface administratively on or off, what's its MTU, what's its MAC.
> It's also the **only one of the three that lets you change something** — `ip link set` is how you actually bring an interface up or down, not just read its state.

**COMMAND:** `ip link`

**PURPOSE (support):** Two jobs. First, a clean read of link-layer state without IP-address noise — useful when you just need "is this interface UP, and what's its MTU" fast. Second, and more important operationally:
> `ip link set` is your lever for administratively **disabling/enabling an interface** — the deliberate, controlled version of "unplug it and plug it back in," usable on a remote box where you can't physically touch a cable.

> `ip addr` and `ip route` are read-only diagnosis. `ip link set` is where you start *changing* things — treat it with the caution that implies.

**SYNTAX:** `ip link [show] [dev INTERFACE]` / `ip link set dev INTERFACE up|down` (often abbreviated `ip l`)

**FLAGS:**

|Flag/Form|Meaning|
|---|---|
|`ip link` / `ip l`|show all interfaces, link-layer info only|
|`ip link show dev eth0`|show just one interface|
|`ip -br link`|brief, one-line-per-interface summary|
|`sudo ip link set dev eth0 down`|administratively **disable** an interface (requires **sudo**)|
|`sudo ip link set dev eth0 up`|administratively **enable** an interface (requires **sudo**)|

**Developer bridge:** think of `ip link set ... down` like pulling a service out of a load balancer's healthy pool — the config and IP are all still there, untouched, but nothing is allowed to flow through it until you put it back `up`. It's a deliberate, reversible switch, not a deletion — same category of action as `docker network disconnect` on a running container.

**EXAMPLE:**

In [ ]:
$ ip link
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN mode DEFAULT group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP mode DEFAULT group default qlen 1000
    link/ether 00:15:5d:c4:9a:1f brd ff:ff:ff:ff:ff:ff

**READ THE OUTPUT:**

- **`<BROADCAST,MULTICAST,UP,LOWER_UP>`** — same flags from the `ip addr` lesson: `UP` = administratively enabled, `LOWER_UP` = physical/virtual link actually active. This is the fastest place to check both without IP clutter.
- **`mtu 1500`** — **Maximum Transmission Unit**, the largest packet size (in bytes) this interface will send without fragmenting. `1500` is the standard Ethernet default. VPNs, tunnels, and some virtual adapters (WSL's included) often run a *lower* MTU to leave room for extra tunnel headers — a mismatch here is a classic cause of "small requests work, large ones hang or time out."
- **`qdisc mq`** — the queueing discipline, i.e. how the kernel schedules outgoing packets on this device. `mq` (multi-queue) is normal for modern NICs; you'll rarely need to touch this, but seeing something unusual here (or `noqueue` on a real interface) is a hint something's non-standard.
- **`state UP`** — the administrative state, spelled out. Matches the `UP` flag above.
- **`link/ether 00:15:5d:c4:9a:1f`** — the MAC address, same as you'd see in `ip addr`.

> Notice what's **missing** compared to `ip addr`: no `inet` lines at all. `ip link` doesn't know or care about IP addresses — that's a layer up. If you only need link-layer facts (state, MTU, MAC), this output is cleaner to scan than wading through `ip addr`'s IP lines to find them.

**TRY NOW:**

Run these in your WSL terminal and paste the real output into a cell below:

```
ip link
ip -br link
```

We'll hold off on `ip link set ... down` for now — that one actually changes state, and it's worth doing deliberately once we cover the gotcha below rather than as a blind "try now."

In [ ]:
dennie@X390:~/linux-tut$ ip link
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN mode DEFAULT group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP mode DEFAULT group default qlen 1000
    link/ether 00:15:5d:ca:53:25 brd ff:ff:ff:ff:ff:ff

In [ ]:
dennie@X390:~/linux-tut$ ip -br link
lo               UNKNOWN        00:00:00:00:00:00 <LOOPBACK,UP,LOWER_UP>
eth0             UP             00:15:5d:ca:53:25 <BROADCAST,MULTICAST,UP,LOWER_UP> 

Healthy on both counts:

- **`eth0` is `UP`** with `mtu 1500` — standard Ethernet default, no VPN/tunnel shrinking it. If a real ticket ever shows something like `mtu 1400` or `1350` here instead, that's your MTU-mismatch lead from the ticket scenario above.
- **`ip -br link`** compresses the same facts into one line per interface: state, MAC, flags — no MTU column though, that's the one thing the brief form drops. If MTU is what you're chasing, you need the full `ip link` (or `ip addr`), not `-br`.
- **MAC differs from the lesson's example** (`00:15:5d:ca:53:25` vs `00:15:5d:c4:9a:1f`) — same story as the IP/gateway drift from earlier lessons: WSL2 regenerates the virtual adapter's MAC too across restarts, not just the IP and gateway. Another one to never hardcode in a runbook.

**TICKET SCENARIO:** "Small requests to this host work fine, but anything that transfers a larger payload — a big API response, a file upload — hangs or times out." This is a textbook **MTU mismatch** symptom: small packets fit under every MTU along the path so they sail through, but a packet that's fine on your box's `1500` gets silently dropped (or needs fragmentation that something along the way blocks) once it crosses a link with a smaller MTU, like a VPN tunnel.

**DIAGNOSIS PATH:**

1. `ip -br link` — quick check: is the interface `UP`, and what's its MTU? Compare against what you'd expect (`1500` for standard Ethernet; lower and consistent if you know a VPN/tunnel is involved).
2. If MTU looks unusual or inconsistent with a known tunnel/VPN configuration, that mismatch is your lead — packets sized for `1500` getting dropped or needing fragmentation on a `1400`-MTU tunnel is a very common "works for small stuff, fails for big stuff" root cause.
3. For a *different* class of ticket — "I need to test what happens when this interface goes down" or "take this NIC offline for maintenance" — `sudo ip link set dev eth0 down` is the controlled way to simulate/perform that, then `sudo ip link set dev eth0 up` to restore it.
4. After any `ip link set` change, re-run `ip link` (or `ip addr`) to confirm the state actually changed the way you expected — don't assume the command succeeded silently.

**GOTCHA:** `ip link set dev eth0 down` requires **sudo**, and if you're connected to the box over SSH *through that same interface*, running it will cut your own connection — you're sawing off the branch you're sitting on. Always double-check which interface an active remote session depends on before bouncing it. On WSL specifically this is lower-stakes (you're on a local console, not remote SSH), but the habit matters — this exact command on a real remote support box is a classic self-inflicted outage.